# 🛠️ ITI AI Track: Native Python Kafka Pipelines Lab

**Objective:** Build an end-to-end event-driven architecture using pure Python clients. You will manage partitions manually, handle direct file serialization into Parquet, and run high-performance SQL analytical queries directly on those files using DuckDB.

---

## 📋 Unified Data Schema Definitions

To keep things clean, all components must strictly follow these schemas:

1. **`topic_raw` / `topic_fraud` JSON Schema:**

```json
{
  "transaction_id": "str",
  "user_id": "int",
  "amount": "float",
  "location": "str"
}
```

2. **`sales_topic` Avro Schema:**

```json
{
  "order_id": "int",
  "item_name": "string",
  "price": "float"
}
```

---

## 🏗️ Environment Setup

### Step 1: Spin Up Infrastructure

Run the following command inside your GitHub Codespace terminal to launch your Docker cluster containing Kafka, Schema Registry, and MinIO:

```bash
docker compose up -d
```

### Step 2: Install Libraries

Install the native Python client libraries needed to interface with the cluster and handle file structures:

```bash
pip install confluent-kafka[avro] fastavro pandas pyarrow duckdb
```

### Step 3: Create Topics Manually

Run commands in your workspace terminal to prepare your message streams with dedicated partition properties:

---

# 🔍 Task 1: Real-Time Fraud Pipeline, Parquet Sinking & DuckDB Analytics

## Step 4: Write the Partition-Targeted Producer (`producer_advanced.py`)

This script acts as the data generator. It sends transaction data to explicit partitions based on a structural business rule.

**Logic**

- If `location == "Cairo"` → send to **Partition 0**
- If `location == "Alexandria"` → send to **Partition 1**

---

## Step 5: Write the Partition-Isolated Consumer (`consumer_partition.py`)

This script mimics an ML scoring engine.

Instead of subscribing to all partitions, it is manually pinned to **Partition 0** only.

**Logic**

- Consume Cairo transactions only.
- If `amount > 100000`, forward the record to `topic_fraud`.

---

## Step 6: Test Offset Strategies (`test_offsets.py`)

Use this script to compare offset behaviors.

### Experiment

- `latest` → ignore historical records.
- `earliest` → replay historical records.

---

## Step 7: Write the Local Parquet Lakehouse Sink (`consumer_to_parquet.py`)

This consumer accumulates fraud alerts and periodically writes them as Parquet files.

---

## Step 8: Run Serverless Analytics via DuckDB (`analytics.py`)

Query all Parquet files directly without Spark or a database server.

---

## Task 2: Python Avro Governance

### Step 9: Write the Local Schema Registry Producer (producer_avro_local.py)

This script introduces client-side data governance using local Python parsing libraries instead of an external server container.

Logic:

- Parse the explicit sales_topic Avro schema string definition within your script using fastavro.parse_schema().

- The Enforcement Step: Before calling the standard Kafka producer, pass your data dictionary through fastavro.validation.validate().

- Serialization Process: If validation passes, use fastavro.schemaless_writer with an in-memory byte buffer (io.BytesIO) to convert the dictionary into compressed binary bytes, then push the raw bytes to Kafka.

Experiment with Local Data Contracts:

- Test Case A (The Valid Run): Pass correct data matching the layout types precisely (e.g., {"order_id": 101, "item_name": "Laptop", "price": 1200.50}). Verify that it serializes and delivers cleanly to Kafka as binary bytes.

- Test Case B (The Broken Schema Run): Attempt to pass an explicit string value directly into the order_id field (e.g., {"order_id": "ABC_NOT_AN_INT", ...}). Run the script and observe how the local validate() function instantly flags the mismatch and raises a validation exception in your terminal, blocking transmission before bad bytes reach the broker network wire.

---

### Step 10: Write the Local Avro Consumer (consumer_avro_local.py)

Because standard Kafka brokers only store and return raw binary arrays without knowing what schema was used, your consumer application must handle deserialization cleanly.

Logic:

- Subscribe to sales_topic using a standard consumer.

- Load the exact same local copy of the sales_topic Avro schema layout.

- Upon polling a message, extract the raw bytes from msg.value(), wrap them in an in-memory byte streams container, and pass them to fastavro.schemaless_reader along with your schema definition to successfully decode the data payload back into a readable Python dictionary.

---

# 🏁 Lab Execution Run-Order Checklist

Open multiple terminal windows and run the components in the following order:

### Terminal 1

```bash
python consumer_to_parquet.py
```

### Terminal 2

```bash
python consumer_partition.py
```

### Terminal 3

```bash
python producer_advanced.py
```

### Terminal 4

After Parquet files appear inside `data_lake/`:

```bash
python analytics.py
```

### Terminal 4 (Next Step)

```bash
python producer_avro.py
```

---

## Expected Flow

```text
Producer
    │
    ▼
topic_raw (2 partitions)
    │
    ▼
Partition-Isolated Consumer (Partition 0 only)
    │
    ▼
topic_fraud
    │
    ▼
Parquet Sink
    │
    ▼
data_lake/*.parquet
    │
    ▼
DuckDB Analytics
```

---
